<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH32_RAG_Part_II_Chatting_with_Your_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU supabase langchain-community
!pip install -qU langchain-groq langchain-text-splitters langchain-core
!pip install -qU langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.1 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from supabase.client import create_client, Client
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_huggingface import HuggingFaceEmbeddings # Updated import
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

# ==========================================
# 1단계: 환경 변수 및 인증 정보 설정
# ==========================================
# 1) Supabase 접속 정보 (앞서 확인한 API 주소와 anon 키)
SUPABASE_URL = "https://tllrshlupjcikzaywaqm.supabase.co"
SUPABASE_KEY = userdata.get('supabase')

supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 2) Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

# ==========================================
# 2단계: 데이터 로드, 분할 및 임베딩 (Ingestion)
# ==========================================
# 문서 로드 및 분할
raw_documents = TextLoader('./test.txt', encoding='utf-8').load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

# 수파베이스와 연동할 HuggingFace 384차원 임베딩 모델 설정
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Supabase VectorStore에 데이터 Store (인덱싱 및 저장)
db = SupabaseVectorStore.from_documents(
    documents=documents,
    embedding=embeddings_model,
    client=supabase_client,
    table_name="documents",
    query_name="match_documents"  # SQL Editor에 등록한 함수 이름
)

# ==========================================
# 3단계: 검색기(Retriever) 및 프롬프트, Groq LLM 설정
# ==========================================
# 유사한(cosine similarity) 문서 2개를 뽑아오는 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k": 2})

query = 'Who are the key figures in the ancient greek history of philosophy?'

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}

    Question: {question}"""
)

# 초고속 가성비 모델인 Groq(Llama 3) 모델로 교체
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)


# ==========================================
# 4단계: LCEL @chain 데코레이터를 활용한 효율적인 파이썬 함수 캡슐화 (Generation)
# ==========================================
print("Running RAG system using Groq and Supabase...\n")

@chain
def qa_chain(input_query):
    # 1. Retrieval: 질문과 관련된 문서 조각 가져오기
    docs = retriever.invoke(input_query)

    # 2. Prompt 조립: 질문과 컨텍스트 결합
    formatted = prompt.invoke({"context": docs, "question": input_query})

    # 3. Generation: Groq LLM을 통해 답변 생성
    answer = llm.invoke(formatted)
    return answer

# RAG 체인 실행 및 결과 출력
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

/tmp/ipykernel_1022/3442127366.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import SupabaseVectorStore


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Running RAG system using Groq and Supabase...

[최종 답변 결과]
Unfortunately, the provided context does not mention specific key figures in ancient Greek philosophy. It only mentions that "Greek philosophers and scientists" laid the groundwork for diverse fields of study, but it does not provide any names or details about individual philosophers.


# **Query Routing**

In [3]:
# @title **Logical Routing**
# In logical routing, we give the LLM knowledge of the various data sources at our
# disposal and then let the LLM reason which data source to apply based on the user's query

In [5]:
from typing import Literal
from pydantic import BaseModel, Field

# 1. 라우팅을 위한 데이터 모델 정의 (Pydantic)
# LLM이 낼 수 있는 답변의 선택지를 딱 두 가지("python_docs" 또는 "js_docs")로 제한
# LLM 구조화(Structured Output)에서 Literal[]이 중요

class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    # 질문이 들어왔을 때 분기할 데이터 소스를 정의합니다.
    # ...**은 이 데이터 소스 항목이 "선택이 아닌 필수(Required)" 무조건 선택하라
    # description은 일반적인 파이썬 코드에서는 주석에 불과하지만,
    # LangChain과 LLM(대형 언어 모델) 세계에서는 인공지능이 직접 읽고 실행하는 '미션 지침서'
    datasource: Literal["python_docs", "js_docs"] = Field(
        ...,
        description="""Given a user question, choose which datasource would be
 most relevant for answering their question""",
    )

# 2. LLM 초기화 및 구조화된 출력(Structured Output) 설정
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)
# Pydantic 모델을 구조화된 출력 대상으로 지정하여 지정된 스키마대로 답변을 받기
structured_llm = llm.with_structured_output(RouteQuery)

# 3. 라우팅 전용 시스템 프롬프트 정의
system = """You are an expert at routing a user question to the appropriate data
 source.
Based on the programming language the question is referring to, route it to the
 relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# 4. 라우터 체인 구성
router = prompt | structured_llm

# 5. 실행 예시 (테스트용)
result = router.invoke({"question": "How do I create a list in Python?"})
print(result.datasource)  # 출력 결과: python_docs

python_docs


In [6]:
result = router.invoke({"question": "자바스크립트에서 화살표 함수는 어떻게 써?"})
print(result.datasource)  # 출력 결과: js_docs

js_docs


In [7]:
question = """Why doesn't the following code work:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""
result = router.invoke({"question": question})
result.datasource

'python_docs'

**router체인의 결과물 result.datasource를 받아 실제로 적절한 목적지(체인)로 데이터 흐름을 갈라지게 만드는 "실행용 분기 코드"**
- Once we’ve extracted the relevant data source, we can pass the value into another function to execute additional logic as required:

- "chain for python_docs"는 파이썬 관련 문서들이 저장된 데이터베이스(Vector Store)를 검색하고 답변을 생성하는 '진짜 LangChain 체인(Chain)' 객체


In [16]:
from langchain_core.runnables import RunnableLambda

def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here
        return "chain for python_docs"
    else:
        ### Logic here
        return "chain for js_docs"

full_chain = router | RunnableLambda(choose_route)

result = full_chain.invoke({"question": question})

In [37]:
new_question = "파이썬에서 리스트 컴프리헨션은 어떻게 사용하나요?"
result_new = full_chain.invoke({"question": new_question})
print(result_new)

chain for python_docs


In [23]:
new_question = "자바스크립트에서 리스트 컴프리헨션은 어떻게 사용하나요?"
result_new = full_chain.invoke({"question": new_question})
print(result_new)

chain for js_docs


In [24]:
# 원인: 사용자가 "자바(Java)"에 대해 질문하면, LLM은 RouteQuery 양식에 맞춰 값을 채우려고 시도합니다. 하지만 선택지에 java_docs 같은 것이 없기 때문에,
# LLM이 강제로 둘 중 하나를 무리하게 선택하다가 형식이 깨지거나 Pydantic의 Validation Error(타입 검증 에러)를 발생시킵니다.

new_question = "자바에서 리스트 컴프리헨션은 어떻게 사용하나요?"
result_new = full_chain.invoke({"question": new_question})
print(result_new)

BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool RouteQuery did not match schema: errors: [`/datasource`: value must be one of "python_docs", "js_docs"]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=RouteQuery> {"name": "RouteQuery", "datasource": "java_docs"} </function>'}}

1. 판단(Routing)을 행동(Execution)으로 연결

- 앞 단계의 router가 사용자의 질문을 분석하여 "python_docs" 또는 "js_docs"라는 판단 결과를 내렸다면, choose_route 함수는 그 결과를 넘겨받아 실제로 파이썬 문서 검색 체인을 실행할지, 자바스크립트 문서 검색 체인을 실행할지 행동 지침을 내려줍니다.

2. LLM의 불확실성 제어 (회복탄력성 확보)
- 텍스트에서 읽었듯이, LLM은 가끔 대소문자를 틀리거나 사소한 오타를 내는 등 무작위성(Random nature)을 가집니다. choose_route 함수는 내부적으로 .lower()와 in 연산자를 사용하여, LLM이 사소한 출력 실수를 하더라도 시스템이 에러로 멈추지 않고 유연하게 대처(Resilience)할 수 있도록 필터링해 줍니다.

3. 조건별 체인 분기 (If/Else 게이트웨이)
- 궁극적으로는 파이프라인 안에서 다음과 같이 작동하며 시스템의 확장성을 담당합니다.


#### 사용자가 파이썬을 물어봄 ➡️ router가 분류 ➡️ choose_route가 파이썬 전용 벡터 스토어/API 체인으로 연결

#### 사용자가 자바스크립트를 물어봄 ➡️ router가 분류 ➡️ choose_route가 자바스크립트 전용 벡터 스토어/API 체인으로 연결

In [27]:
# @title **Semantic Routing**

# Two prompts
physics_template = """You are a very smart physics professor. You are great at
 answering questions about physics in a concise and easy-to-understand manner.
 When you don't know the answer to a question, you admit that you don't know.
Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering
 math questions. You are so good because you are able to break down hard
 problems into their component parts, answer the component parts, and then
 put them together to answer the broader question.
Here is a question:
{query}"""

# Embed prompts, # 수파베이스와 연동할 HuggingFace 384차원 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [42]:
import numpy as np
print(f'프롬프트 개수 : {len(prompt_embeddings)} 개')
print(f'임베딩 차원은 : {len(prompt_embeddings[0])}')
dot_product = np.dot(prompt_embeddings[0], prompt_embeddings[1])
print(f"prompt_embeddings[0]과 prompt_embeddings[1]의 내적: {dot_product}")

프롬프트 개수 : 2 개
임베딩 차원은 : 384
prompt_embeddings[0]과 prompt_embeddings[1]의 내적: 0.5979606230558369


In [43]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x79db84340a40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x79db84420a40>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [49]:
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [51]:
query_embedding = embeddings.embed_query("What's a black hole")
print(f'질문(query)의 embwdding vector 차원: {len(query_embedding)}')

질문(query)의 embwdding vector 차원: 384


In [54]:
# query_embedding에 리스트를 ?
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html   요기요!
cosine_similarity([query_embedding], prompt_embeddings)

array([[0.18705176, 0.10095931]])

In [58]:
# 코사인 유사도 계산: "블랙홀" 벡터가 앞서 만든 "물리 프롬프트 벡터"와 "수학 프롬프트 벡터" 중 어느 쪽에 공간적으로 더 가까운지 각도를 계산하면
# similarity.argmax(): 가장 높은 유사도 점수를 가진 인덱스(방향)를 찾기
# 당연히 "블랙홀"이라는 단어의 의미적 특성상 math_template 보다는 physics_template 쪽 점수가 압도적으로 높게 나오게 됨
# 시스템은 점수가 가장 높은 physics_template을 최종 목적지로 채택

similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
similarity.argmax()

np.int64(0)

In [60]:
prompt_templates

["You are a very smart physics professor. You are great at\n answering questions about physics in a concise and easy-to-understand manner.\n When you don't know the answer to a question, you admit that you don't know.\nHere is a question:\n{query}",
 'You are a very good mathematician. You are great at answering\n math questions. You are so good because you are able to break down hard\n problems into their component parts, answer the component parts, and then\n put them together to answer the broader question.\nHere is a question:\n{query}']

In [64]:
# prompt_templates에서 물리학과 관련 있는 첫번째 0번을 가져와 사용자 질문과 합쳐서 prompt를 완성하여 llm 과 StrOutputParser() 출력양식으로 runnable chain LCEL
most_similar = prompt_templates[similarity.argmax()]
most_similar

"You are a very smart physics professor. You are great at\n answering questions about physics in a concise and easy-to-understand manner.\n When you don't know the answer to a question, you admit that you don't know.\nHere is a question:\n{query}"

In [65]:
from langchain_core.runnables import chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sklearn.metrics.pairwise import cosine_similarity

# Route question to prompt
@chain
def prompt_router(query):
    # Embed question
    query_embedding = embeddings.embed_query(query)

    # Compute similarity
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]

    # Pick the prompt most similar to the input question
    most_similar = prompt_templates[similarity.argmax()]

    return PromptTemplate.from_template(most_similar)

semantic_router = (
 prompt_router
 | llm
 | StrOutputParser()
)

print(semantic_router.invoke("What's a black hole"))

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime around it.

Imagine spacetime as a trampoline. If you place a heavy object, like a bowling ball, on the trampoline, it will warp and curve, creating a dent. That's similar to what happens with a black hole, but instead of a bowling ball, it's a massive star that's collapsed.

The point of no return, called the event horizon, marks the boundary of the black hole. Once something crosses the event horizon, it's trapped by the black hole's gravity and can't escape. That's why black holes are invisible, even though they're massive objects – not even light can escape to reach our eyes.

Black holes come in different sizes, ranging from small, stellar-mass black holes formed from the collapse of individual stars, to supermassive black holes found at t

In [70]:
print(semantic_router.invoke("L2 reguralization은 무엇이지? 아주 간단하게 답변해"))

L2 정규화(L2 regularization)는 머신 러닝에서 사용되는 정규화 기법입니다. 

정규화란 모델의 복잡성을 제한하여 과적합을 방지하는 것입니다. L2 정규화는 모델의 가중치(w)가 0에 가까운 값을 갖도록 유도하여 과적합을 방지합니다.

L2 정규화의 공식은 다음과 같습니다.

L2 = (1/2) * w^2

여기서 w는 모델의 가중치입니다. L2 정규화의 목적은 모델의 가중치를 0에 가까운 값을 갖도록 유도하여 과적합을 방지하는 것입니다.


In [ ]:
print(semantic_router.invoke("암흑물질은 무엇이지? 아주 간단하게 답변해"))